<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/risk_classifier_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# CELL 1 — GOOGLE DRIVE + TRAINING PATH SETUP
# MODEL 02 — COMMAND RISK CLASSIFIER
# =========================================================

from google.colab import drive

import os


# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

DRIVE_MOUNT = "/content/drive"

drive.mount(
    DRIVE_MOUNT,
    force_remount=False
)

assert os.path.exists(
    f"{DRIVE_MOUNT}/MyDrive"
), "Google Drive belum mounted"


# =========================================================
# 2. STORAGE DIRECTORY
# =========================================================

SAVE_DIR = (
    f"{DRIVE_MOUNT}/MyDrive/command_risk"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


# =========================================================
# 3. SUBDIRECTORIES
# =========================================================

RAW_DIR = (
    f"{SAVE_DIR}/raw"
)

PROCESSED_DIR = (
    f"{SAVE_DIR}/processed"
)

TEACHER_DIR = (
    f"{SAVE_DIR}/teacher"
)

TRUSTED_DIR = (
    f"{SAVE_DIR}/trusted"
)

SYNTHETIC_DIR = (
    f"{SAVE_DIR}/synthetic"
)

REPAIR_DIR = (
    f"{SAVE_DIR}/repair"
)

BLIND_TEST_DIR = (
    f"{SAVE_DIR}/blind_test"
)

MODEL_DIR = (
    f"{SAVE_DIR}/models"
)

ONNX_DIR = (
    f"{MODEL_DIR}/onnx"
)

REPORT_DIR = (
    f"{SAVE_DIR}/reports"
)

FEATURE_DIR = (
    f"{SAVE_DIR}/features"
)


for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    TEACHER_DIR,
    TRUSTED_DIR,
    SYNTHETIC_DIR,
    REPAIR_DIR,
    BLIND_TEST_DIR,
    MODEL_DIR,
    ONNX_DIR,
    REPORT_DIR,
    FEATURE_DIR,
]:

    os.makedirs(
        directory,
        exist_ok=True
    )


# =========================================================
# 4. MODEL 01 SOURCE PATHS
# =========================================================

ACTION_CLASSIFIER_DIR = (
    f"{DRIVE_MOUNT}/MyDrive/action_classifier"
)

ACTION_TRUSTED_PATH = (
    f"{ACTION_CLASSIFIER_DIR}/trusted/"
    "action_classifier_trusted_v1.jsonl"
)

ACTION_MODEL_DIR = (
    f"{ACTION_CLASSIFIER_DIR}/models"
)

ACTION_ONNX_PATH = (
    f"{ACTION_MODEL_DIR}/onnx/"
    "action_classifier.onnx"
)


# =========================================================
# 5. COMMAND RISK DATASET PATHS
# =========================================================

RAW_CORPUS_PATH = (
    f"{RAW_DIR}/command_risk_raw_v1.jsonl"
)

NORMALIZED_DATASET_PATH = (
    f"{PROCESSED_DIR}/normalized_dataset_v1.jsonl"
)

DEDUP_DATASET_PATH = (
    f"{PROCESSED_DIR}/dedup_dataset_v1.jsonl"
)

STATIC_FEATURES_PATH = (
    f"{FEATURE_DIR}/static_features_v1.jsonl"
)

ACTION_FEATURES_PATH = (
    f"{FEATURE_DIR}/action_features_v1.jsonl"
)

TEACHER_LABELED_PATH = (
    f"{TEACHER_DIR}/teacher_labeled_v1.jsonl"
)

TEACHER_REJECTED_PATH = (
    f"{TEACHER_DIR}/teacher_rejected_v1.jsonl"
)

SYNTHETIC_DATASET_PATH = (
    f"{SYNTHETIC_DIR}/synthetic_risk_v1.jsonl"
)

TRUSTED_TRAIN_PATH = (
    f"{TRUSTED_DIR}/command_risk_trusted_v1.jsonl"
)

REPAIR_DATASET_PATH = (
    f"{REPAIR_DIR}/targeted_repair_v1.jsonl"
)

BLIND_TEST_PATH = (
    f"{BLIND_TEST_DIR}/blind_test_v1.jsonl"
)


# =========================================================
# 6. MODEL OUTPUT PATHS
# =========================================================

MODEL_PATH = (
    f"{MODEL_DIR}/command_risk.joblib"
)

BEST_MODEL_PATH = (
    f"{MODEL_DIR}/command_risk_best.joblib"
)

VECTORIZER_PATH = (
    f"{MODEL_DIR}/command_risk_vectorizer.joblib"
)

FEATURE_CONFIG_PATH = (
    f"{MODEL_DIR}/feature_config.json"
)

ONNX_MODEL_PATH = (
    f"{ONNX_DIR}/command_risk.onnx"
)


# =========================================================
# 7. REPORT PATHS
# =========================================================

METRICS_PATH = (
    f"{REPORT_DIR}/training_metrics.json"
)

PREDICTIONS_PATH = (
    f"{REPORT_DIR}/test_predictions.csv"
)

ERROR_ANALYSIS_PATH = (
    f"{REPORT_DIR}/error_analysis.csv"
)

LABEL_DISTRIBUTION_PATH = (
    f"{REPORT_DIR}/label_distribution.csv"
)

CONFUSION_MATRIX_PATH = (
    f"{REPORT_DIR}/confusion_matrix.csv"
)

ONNX_PARITY_PATH = (
    f"{REPORT_DIR}/onnx_parity.json"
)

BENCHMARK_PATH = (
    f"{REPORT_DIR}/latency_benchmark.json"
)


# =========================================================
# 8. BASIC VALIDATION
# =========================================================

assert os.path.exists(
    SAVE_DIR
)

assert os.path.exists(
    MODEL_DIR
)

assert os.path.exists(
    REPORT_DIR
)


# =========================================================
# 9. STATUS
# =========================================================

print()
print("=" * 70)
print("MODEL 02 — COMMAND RISK CLASSIFIER STORAGE")
print("=" * 70)

print(
    "SAVE_DIR          :",
    SAVE_DIR
)

print(
    "Model 01 corpus   :",
    ACTION_TRUSTED_PATH
)

print(
    "Raw corpus        :",
    RAW_CORPUS_PATH
)

print(
    "Teacher labeled   :",
    TEACHER_LABELED_PATH
)

print(
    "Synthetic data    :",
    SYNTHETIC_DATASET_PATH
)

print(
    "Trusted train     :",
    TRUSTED_TRAIN_PATH
)

print(
    "Repair dataset    :",
    REPAIR_DATASET_PATH
)

print(
    "Best model        :",
    BEST_MODEL_PATH
)

print(
    "ONNX model        :",
    ONNX_MODEL_PATH
)

print(
    "Metrics           :",
    METRICS_PATH
)

print("=" * 70)

In [ ]:
# =========================================================
# CELL 2 — DEPENDENCIES + IMPORTS
# MODEL 02 — COMMAND RISK CLASSIFIER
# =========================================================

!pip install -q \
    pandas \
    numpy \
    scipy \
    scikit-learn \
    joblib \
    requests \
    tqdm


# =========================================================
# 1. STANDARD LIBRARY
# =========================================================

import os
import re
import json
import time
import random
import hashlib
import shlex

from pathlib import Path
from collections import Counter, defaultdict


# =========================================================
# 2. DATA
# =========================================================

import numpy as np
import pandas as pd


# =========================================================
# 3. SCIKIT-LEARN
# =========================================================

from sklearn.model_selection import (
    train_test_split,
)

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
)

from sklearn.preprocessing import (
    StandardScaler,
)

from sklearn.linear_model import (
    LogisticRegression,
)

from sklearn.svm import (
    LinearSVC,
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

from scipy.sparse import (
    csr_matrix,
    hstack,
)

import joblib


# =========================================================
# 4. UTILITIES
# =========================================================

import requests

from tqdm.auto import tqdm


# =========================================================
# 5. RANDOM SEED
# =========================================================

RANDOM_SEED = 42

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)


# =========================================================
# 6. PROJECT CONFIG
# =========================================================

MODEL_NAME = "command_risk_classifier"

MODEL_VERSION = "v1"


# =========================================================
# 7. STATUS
# =========================================================

print()
print("=" * 70)
print("MODEL 02 — DEPENDENCIES READY")
print("=" * 70)

print("Model       :", MODEL_NAME)
print("Version     :", MODEL_VERSION)
print("Random seed :", RANDOM_SEED)
print("SAVE_DIR    :", SAVE_DIR)

print("=" * 70)

In [ ]:
# =========================================================
# CELL 3 — RISK TAXONOMY + FEATURE SCHEMA
# MODEL 02 — COMMAND RISK CLASSIFIER
# =========================================================


# =========================================================
# 1. RISK LABELS
# =========================================================

RISK_LABELS = [
    "BENIGN",
    "SUSPICIOUS",
    "DESTRUCTIVE",
]


# =========================================================
# 2. RISK TAXONOMY
# =========================================================

RISK_TAXONOMY = {

    "BENIGN": {
        "description": (
            "Low-risk shell command. Primarily reads information, "
            "performs ordinary local operations, or makes bounded "
            "changes without clear destructive intent."
        ),

        "examples": [
            "ls -la",
            "pwd",
            "cat /etc/os-release",
            "git status",
            "mkdir test_dir",
            "cp test.txt backup.txt",
            "python app.py",
        ],
    },

    "SUSPICIOUS": {
        "description": (
            "Potentially risky command that may still be legitimate. "
            "Requires additional review because it contains privilege "
            "use, network-to-execution behavior, shell spawning, "
            "encoded payloads, sensitive paths, command substitution, "
            "or other high-risk execution patterns."
        ),

        "examples": [
            "sudo systemctl restart nginx",
            "curl <TEST_URL> | bash",
            "wget <TEST_URL> -O- | sh",
            "echo <BASE64_DATA> | base64 -d | bash",
            "chmod +x script.sh && ./script.sh",
        ],
    },

    "DESTRUCTIVE": {
        "description": (
            "Command clearly intended or highly likely to cause "
            "irreversible or major damage such as deleting critical "
            "files, overwriting storage, formatting devices, or "
            "destroying system state."
        ),

        "examples": [
            "rm -rf /",
            "rm -rf /*",
            "mkfs.ext4 /dev/sda",
            "dd if=/dev/zero of=/dev/sda",
            "find / -delete",
        ],
    },
}


# =========================================================
# 3. ACTION CLASSIFIER LABELS
# =========================================================

ACTION_LABELS = [
    "READ",
    "WRITE",
    "DELETE",
    "EXECUTE",
    "NETWORK",
    "INSTALL",
    "PRIVILEGED",
    "SYSTEM_CHANGE",
]


# =========================================================
# 4. STATIC FEATURE SCHEMA
# =========================================================

STATIC_FEATURE_NAMES = [
    "sudo_present",
    "pipe_present",
    "redirect_present",

    "curl_present",
    "wget_present",

    "chmod_present",
    "eval_present",
    "base64_present",

    "recursive_delete",
    "wildcard_delete",

    "systemctl_present",
    "shell_spawn",

    "credential_path",
    "system_path",

    "network_access",

    # tambahan untuk chained-command analysis
    "command_chain_present",
    "command_substitution",
]


# =========================================================
# 5. VALIDATION
# =========================================================

assert len(RISK_LABELS) == len(set(RISK_LABELS))
assert len(ACTION_LABELS) == len(set(ACTION_LABELS))
assert len(STATIC_FEATURE_NAMES) == len(
    set(STATIC_FEATURE_NAMES)
)


# =========================================================
# 6. STATUS
# =========================================================

print()
print("=" * 70)
print("MODEL 02 — RISK TAXONOMY")
print("=" * 70)

for label in RISK_LABELS:

    print()
    print(f"[{label}]")

    print(
        RISK_TAXONOMY[label]["description"]
    )


print()
print("=" * 70)

print(
    "Risk labels     :",
    len(RISK_LABELS)
)

print(
    "Action labels   :",
    len(ACTION_LABELS)
)

print(
    "Static features :",
    len(STATIC_FEATURE_NAMES)
)

print("=" * 70)

In [ ]:
# =========================================================
# CELL 3 — RISK TAXONOMY + FEATURE SCHEMA
# MODEL 02 — COMMAND RISK CLASSIFIER
# =========================================================


# =========================================================
# 1. RISK LABELS
# =========================================================

RISK_LABELS = [
    "BENIGN",
    "SUSPICIOUS",
    "DESTRUCTIVE",
]


# =========================================================
# 2. RISK TAXONOMY
# =========================================================

RISK_TAXONOMY = {

    "BENIGN": {
        "description": (
            "Low-risk shell command. Primarily reads information, "
            "performs ordinary local operations, or makes bounded "
            "changes without clear destructive intent."
        ),

        "examples": [
            "ls -la",
            "pwd",
            "cat /etc/os-release",
            "git status",
            "mkdir test_dir",
            "cp test.txt backup.txt",
            "python app.py",
        ],
    },

    "SUSPICIOUS": {
        "description": (
            "Potentially risky command that may still be legitimate. "
            "Requires additional review because it contains privilege "
            "use, network-to-execution behavior, shell spawning, "
            "encoded payloads, sensitive paths, command substitution, "
            "or other high-risk execution patterns."
        ),

        "examples": [
            "sudo systemctl restart nginx",
            "curl <TEST_URL> | bash",
            "wget <TEST_URL> -O- | sh",
            "echo <BASE64_DATA> | base64 -d | bash",
            "chmod +x script.sh && ./script.sh",
        ],
    },

    "DESTRUCTIVE": {
        "description": (
            "Command clearly intended or highly likely to cause "
            "irreversible or major damage such as deleting critical "
            "files, overwriting storage, formatting devices, or "
            "destroying system state."
        ),

        "examples": [
            "rm -rf /",
            "rm -rf /*",
            "mkfs.ext4 /dev/sda",
            "dd if=/dev/zero of=/dev/sda",
            "find / -delete",
        ],
    },
}


# =========================================================
# 3. ACTION CLASSIFIER LABELS
# =========================================================

ACTION_LABELS = [
    "READ",
    "WRITE",
    "DELETE",
    "EXECUTE",
    "NETWORK",
    "INSTALL",
    "PRIVILEGED",
    "SYSTEM_CHANGE",
]


# =========================================================
# 4. STATIC FEATURE SCHEMA
# =========================================================

STATIC_FEATURE_NAMES = [
    "sudo_present",
    "pipe_present",
    "redirect_present",

    "curl_present",
    "wget_present",

    "chmod_present",
    "eval_present",
    "base64_present",

    "recursive_delete",
    "wildcard_delete",

    "systemctl_present",
    "shell_spawn",

    "credential_path",
    "system_path",

    "network_access",

    # tambahan untuk chained-command analysis
    "command_chain_present",
    "command_substitution",
]


# =========================================================
# 5. VALIDATION
# =========================================================

assert len(RISK_LABELS) == len(set(RISK_LABELS))
assert len(ACTION_LABELS) == len(set(ACTION_LABELS))
assert len(STATIC_FEATURE_NAMES) == len(
    set(STATIC_FEATURE_NAMES)
)


# =========================================================
# 6. STATUS
# =========================================================

print()
print("=" * 70)
print("MODEL 02 — RISK TAXONOMY")
print("=" * 70)

for label in RISK_LABELS:

    print()
    print(f"[{label}]")

    print(
        RISK_TAXONOMY[label]["description"]
    )


print()
print("=" * 70)

print(
    "Risk labels     :",
    len(RISK_LABELS)
)

print(
    "Action labels   :",
    len(ACTION_LABELS)
)

print(
    "Static features :",
    len(STATIC_FEATURE_NAMES)
)

print("=" * 70)

In [ ]:
# =========================================================
# CELL 4 — STATIC FEATURE EXTRACTOR
# MODEL 02 — COMMAND RISK CLASSIFIER
# =========================================================


# =========================================================
# 1. HELPER
# =========================================================

def regex_present(pattern, command):

    return int(
        re.search(
            pattern,
            command,
            flags=re.IGNORECASE,
        )
        is not None
    )


# =========================================================
# 2. FEATURE EXTRACTOR
# =========================================================

def extract_static_features(command):

    command = str(command).strip()

    command_lower = command.lower()


    features = {}


    # -----------------------------------------------------
    # PRIVILEGE
    # -----------------------------------------------------

    features["sudo_present"] = regex_present(
        r"(^|\s|[;&|])sudo(\s|$)",
        command,
    )


    # -----------------------------------------------------
    # SHELL COMPOSITION
    # -----------------------------------------------------

    features["pipe_present"] = int(
        "|" in command
    )

    features["redirect_present"] = int(
        bool(
            re.search(
                r"(>>?|<<?)",
                command,
            )
        )
    )

    features["command_chain_present"] = int(
        bool(
            re.search(
                r"(&&|\|\||;)",
                command,
            )
        )
    )

    features["command_substitution"] = int(
        "$(" in command
        or
        bool(
            re.search(
                r"`[^`]+`",
                command,
            )
        )
    )


    # -----------------------------------------------------
    # NETWORK
    # -----------------------------------------------------

    features["curl_present"] = regex_present(
        r"(^|\s|[;&|])curl(\s|$)",
        command,
    )

    features["wget_present"] = regex_present(
        r"(^|\s|[;&|])wget(\s|$)",
        command,
    )

    features["network_access"] = int(
        features["curl_present"]
        or
        features["wget_present"]
        or
        bool(
            re.search(
                r"\bhttps?://",
                command_lower,
            )
        )
        or
        regex_present(
            r"(^|\s)(ssh|scp|sftp|nc|ncat|netcat)(\s|$)",
            command,
        )
    )


    # -----------------------------------------------------
    # EXECUTION
    # -----------------------------------------------------

    features["chmod_present"] = regex_present(
        r"(^|\s|[;&|])chmod(\s|$)",
        command,
    )

    features["eval_present"] = regex_present(
        r"(^|\s|[;&|])eval(\s|$)",
        command,
    )

    features["shell_spawn"] = int(
        bool(
            re.search(
                r"(^|\s|[;&|])"
                r"(bash|sh|zsh|dash|ksh)"
                r"(\s|$)",
                command_lower,
            )
        )
    )


    # -----------------------------------------------------
    # ENCODING / OBFUSCATION
    # -----------------------------------------------------

    features["base64_present"] = regex_present(
        r"(^|\s|[;&|])base64(\s|$)",
        command,
    )


    # -----------------------------------------------------
    # DELETE PATTERNS
    # -----------------------------------------------------

    features["recursive_delete"] = int(
        bool(
            re.search(
                r"\brm\s+[^;&|]*-[a-z]*r[a-z]*f?",
                command_lower,
            )
        )
        or
        bool(
            re.search(
                r"\brm\s+[^;&|]*-[a-z]*f[a-z]*r",
                command_lower,
            )
        )
    )

    features["wildcard_delete"] = int(
        bool(
            re.search(
                r"\brm\b[^;&|]*[\*\?]",
                command_lower,
            )
        )
    )


    # -----------------------------------------------------
    # SYSTEM
    # -----------------------------------------------------

    features["systemctl_present"] = regex_present(
        r"(^|\s|[;&|])systemctl(\s|$)",
        command,
    )


    # -----------------------------------------------------
    # SENSITIVE PATHS
    # -----------------------------------------------------

    credential_patterns = [
        r"/\.ssh(?:/|\s|$)",
        r"\.ssh/",
        r"id_rsa",
        r"id_ed25519",
        r"\.aws/credentials",
        r"\.config/gcloud",
        r"/etc/shadow",
        r"/etc/passwd",
    ]

    features["credential_path"] = int(
        any(
            re.search(
                pattern,
                command_lower,
            )
            for pattern in credential_patterns
        )
    )


    system_patterns = [
        r"/etc/",
        r"/boot/",
        r"/usr/",
        r"/bin/",
        r"/sbin/",
        r"/lib/",
        r"/lib64/",
        r"/var/lib/",
        r"/dev/",
        r"/proc/",
        r"/sys/",
    ]

    features["system_path"] = int(
        any(
            re.search(
                pattern,
                command_lower,
            )
            for pattern in system_patterns
        )
    )


    # -----------------------------------------------------
    # ORDER VALIDATION
    # -----------------------------------------------------

    features = {
        name: int(
            features.get(
                name,
                0,
            )
        )
        for name in STATIC_FEATURE_NAMES
    }


    return features


# =========================================================
# 3. VECTOR CONVERSION
# =========================================================

def static_features_to_vector(features):

    return np.array(
        [
            features[name]
            for name in STATIC_FEATURE_NAMES
        ],
        dtype=np.float32,
    )


# =========================================================
# 4. SANITY TEST
# =========================================================

STATIC_TEST_COMMANDS = [
    "ls -la",
    "sudo systemctl restart nginx",
    "curl <TEST_URL> | bash",
    "rm -rf /tmp/test",
    "rm -rf /*",
    "cat ~/.ssh/id_ed25519",
    "echo test > /etc/test.conf",
]


for command in STATIC_TEST_COMMANDS:

    features = extract_static_features(
        command
    )

    active = [
        name
        for name, value
        in features.items()
        if value == 1
    ]

    print()
    print("COMMAND:")
    print(command)

    print(
        "FEATURES:",
        active
    )

In [ ]:
# =========================================================
# CELL 5 — LOAD TRUSTED CORPUS FROM MODEL 01
# MODEL 02 — COMMAND RISK CLASSIFIER
# =========================================================


# =========================================================
# 1. CHECK SOURCE
# =========================================================

print()
print("=" * 70)
print("LOADING MODEL 01 TRUSTED CORPUS")
print("=" * 70)

print(
    "Source:",
    ACTION_TRUSTED_PATH
)


assert os.path.exists(
    ACTION_TRUSTED_PATH
), (
    "Model 01 trusted dataset tidak ditemukan:\n"
    f"{ACTION_TRUSTED_PATH}"
)


# =========================================================
# 2. JSONL LOADER
# =========================================================

def load_jsonl(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(
            f,
            start=1,
        ):

            line = line.strip()

            if not line:
                continue

            try:

                rows.append(
                    json.loads(line)
                )

            except Exception as exc:

                print(
                    f"Invalid JSON line "
                    f"{line_number}: {exc}"
                )

    return rows


# =========================================================
# 3. LOAD
# =========================================================

action_rows = load_jsonl(
    ACTION_TRUSTED_PATH
)

print()
print(
    "Rows loaded:",
    len(action_rows)
)


assert len(action_rows) > 0, (
    "Trusted corpus Model 01 kosong"
)


# =========================================================
# 4. INSPECT SCHEMA
# =========================================================

print()
print("=" * 70)
print("SAMPLE RECORD")
print("=" * 70)

print(
    json.dumps(
        action_rows[0],
        indent=2,
        ensure_ascii=False,
    )
)


print()
print("=" * 70)
print("AVAILABLE KEYS")
print("=" * 70)

all_keys = sorted(
    {
        key
        for row in action_rows
        for key in row.keys()
    }
)

for key in all_keys:
    print("-", key)


# =========================================================
# 5. COMMAND FIELD DETECTION
# =========================================================

COMMAND_FIELD_CANDIDATES = [
    "command",
    "cmd",
    "text",
    "input",
    "description",
]


COMMAND_FIELD = None


for candidate in COMMAND_FIELD_CANDIDATES:

    if candidate in all_keys:

        COMMAND_FIELD = candidate

        break


assert COMMAND_FIELD is not None, (
    "Tidak menemukan field command. "
    "Periksa SAMPLE RECORD di atas."
)


print()
print(
    "Detected command field:",
    COMMAND_FIELD
)


# =========================================================
# 6. NORMALIZE INTO MODEL 02 RAW CORPUS
# =========================================================

risk_raw_rows = []


for index, row in enumerate(
    action_rows
):

    command = str(
        row.get(
            COMMAND_FIELD,
            "",
        )
    ).strip()


    if not command:
        continue


    item = {
        "id": (
            f"model01_{index:06d}"
        ),

        "command": command,

        "source": (
            "model01_trusted"
        ),

        "model01_record": row,
    }


    risk_raw_rows.append(
        item
    )


# =========================================================
# 7. SAVE RAW CORPUS
# =========================================================

with open(
    RAW_CORPUS_PATH,
    "w",
    encoding="utf-8",
) as f:

    for row in risk_raw_rows:

        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
            )
            + "\n"
        )


# =========================================================
# 8. SUMMARY
# =========================================================

print()
print("=" * 70)
print("MODEL 02 RAW CORPUS READY")
print("=" * 70)

print(
    "Model 01 rows :",
    len(action_rows)
)

print(
    "Usable rows   :",
    len(risk_raw_rows)
)

print(
    "Command field :",
    COMMAND_FIELD
)

print(
    "Saved to      :",
    RAW_CORPUS_PATH
)

print("=" * 70)